# Day 26: Observability & Monitoring

**Week 4 — System Design**

---

## 📚 Theory: The Three Pillars of Observability

In a monolithic architecture, debugging is easy: you SSH into the server and read the logs. In a distributed microservices architecture (like an Agent Orchestrator), a single user request might travel through an API Gateway, a billing service, an Orchestrator worker, Redis, a Tool worker, and an LLM provider. If it fails, where did it fail?

To solve this, system design relies on the **Three Pillars of Observability**:
1. **Logs**: Immutable records of discrete events (e.g., "User logged in at 10:04 AM").
2. **Metrics**: Numerical representations of data measured over time (e.g., "CPU utilization is at 85%", "Error rate is 2%").
3. **Traces**: A representation of the end-to-end journey of a single request across a distributed system.


## 🔨 Exercise: Designing an Observability Stack

### 1. Distributed Tracing
**The Problem**: A developer complains that their Agent execution is taking 45 seconds and timing out. How do you find the bottleneck?

**The Solution (Trace IDs)**:
When a request hits the API Gateway, the gateway generates a unique `TraceID` (e.g., `trace_12345`). This ID is passed in the HTTP Headers (like `X-B3-TraceId`) to every subsequent microservice that touches the request.

As each microservice does its work, it generates a `SpanID` (a sub-unit of work, like "Database Query" or "LLM Call"). It records the start and end time of that span, and sends this data (along with the `TraceID`) asynchronously to a central tracing backend (like Jaeger, Zipkin, or Google Cloud Trace).

You can then pull up `trace_12345` in a dashboard and visually see a Gantt chart of exactly where the 45 seconds were spent (e.g., 2 seconds in the API, 40 seconds waiting for the LLM, 3 seconds executing a tool).

### 2. Metrics Aggregation
Metrics are crucial for triggering alerts. 

Instead of logging every single HTTP 200 (which is expensive and slow), servers expose an endpoint (like `/metrics`). A time-series database (like **Prometheus**) periodically "scrapes" these endpoints to collect aggregate data:
- Request count
- Error count (HTTP 5xx)
- Request latency (p50, p90, p99 percentiles)

This data is then visualized in dashboards (like **Grafana**).

### 3. SLOs, SLIs, and SLAs (The Google SRE Model)

Google pioneered Site Reliability Engineering (SRE). You must know these acronyms:

- **SLI (Service Level Indicator)**: A carefully chosen quantitative measure of some aspect of the level of service that is provided. 
  *Example*: The proportion of HTTP requests that completed successfully.
- **SLO (Service Level Objective)**: A target value or range of values for a service level that is measured by an SLI. 
  *Example*: 99.9% of HTTP requests must complete successfully.
- **SLA (Service Level Agreement)**: An explicit or implicit contract with your users that includes consequences of meeting (or missing) the SLOs they contain. 
  *Example*: If uptime falls below 99.9%, we will refund customers 10% of their monthly bill.

**Error Budgets**:
If your SLO is 99.9% uptime, you have an "Error Budget" of 0.1% downtime (about 43 minutes per month). 
If your development team launches buggy features and uses up all 43 minutes of downtime in the first week, *feature launches are frozen* until the end of the month, and the team must focus entirely on reliability. This balances feature velocity with system stability.

## 📝 Day 26 Review

### Concepts Validated Today
- [ ] The differences between Logs, Metrics, and Traces.
- [ ] Using `TraceID` and `SpanID` to track requests across decoupled microservices.
- [ ] How Time-Series Databases (Prometheus) efficiently scrape aggregate metrics.
- [ ] The Google SRE principles of SLIs, SLOs, SLAs, and Error Budgets.

### Tomorrow's Preview
**Day 27: Cloud Deployment Patterns** — We will explore how code actually gets from a developer's laptop to production, covering Docker, Kubernetes, CI/CD, and deployment strategies like Blue/Green and Canary.